# Introdução e objetivo do pré-processamento

Este notebook tem como objetivo realizar o pré-processamento das bases de dados utilizadas na análise, preparando os dados eleitorais e municipais para as etapas posteriores. O tratamento envolve a padronização das variáveis, correção de inconsistências, remoção de informações irrelevantes, tratamento de valores ausentes e conversão dos atributos para formatos adequados. Essas etapas são necessárias para garantir maior qualidade dos dados e evitar que problemas estruturais das bases originais afetem os resultados das análises futuras.

### Importação das bibliotecas e carregamento dos dados

Inicialmente são importadas as bibliotecas necessárias para manipulação e análise dos dados, principalmente ferramentas voltadas para processamento de tabelas e operações numéricas. Em seguida, são carregadas as bases utilizadas no estudo: a base contendo os dados eleitorais dos candidatos e a base contendo informações municipais. Durante a leitura dos arquivos, são definidos parâmetros para garantir a correta interpretação dos dados, como separadores, formatos de leitura e tratamento inicial de valores ausentes.

In [1]:
import pandas as pd
import numpy as np

votacao = pd.read_csv("votacao_candidato.csv", sep=";", encoding="latin1")

panorama = pd.read_csv("dados_municipios_panorama.csv")



### Padronização inicial dos valores ausentes

Nesta etapa é realizada a identificação e padronização dos valores ausentes presentes nas bases. Diferentes formas de representação de dados inexistentes, como campos vazios, símbolos ou textos indicando ausência de informação, são convertidas para um formato único de valor nulo. Essa padronização permite que as próximas etapas de limpeza e imputação sejam aplicadas de forma consistente em todas as variáveis.

### Filtragem dos dados eleitorais

A base eleitoral inicialmente contém informações referentes a diversos cargos e situações eleitorais. Para manter apenas os registros relevantes ao objetivo do estudo, os dados são filtrados considerando somente os cargos de interesse, incluindo Presidente, Governador, Senador, Deputado Federal, Deputado Estadual e Deputado Distrital. Essa filtragem reduz o volume de dados analisados e garante que as informações utilizadas estejam alinhadas com a proposta da análise.


In [2]:
TARGET_CARGOS = [
    "PRESIDENTE",
    "GOVERNADOR",
    "SENADOR",
    "DEPUTADO ESTADUAL",
    "DEPUTADO FEDERAL",
    "DEPUTADO DISTRITAL"
]

MISSING_TOKENS = [
    "-",
    "Sem dados",
    "",
    "nan",
    "NaN",
    "NAN"
]


def standardize_missing_values(df):
    return df.replace(MISSING_TOKENS, np.nan)


def clean_string_columns(df):
    string_columns = df.select_dtypes(include=["object", "string"]).columns
    for col in string_columns:
        df[col] = df[col].astype(str).str.strip()
        df[col] = df[col].replace({"nan": np.nan, "NaN": np.nan, "NAN": np.nan, "": np.nan})
    return df


def clean_numeric_series(series):
    cleaned = (series.astype(str).str.strip().str.replace(".", "", regex=False).str.replace(",", ".", regex=False))
    return pd.to_numeric(cleaned, errors="coerce")


votacao = standardize_missing_values(votacao)
votacao.columns = votacao.columns.str.strip()
votacao = clean_string_columns(votacao)

votacao = votacao[votacao["Cargo"].astype(str).str.upper().isin(TARGET_CARGOS)].copy()

print("Rows after cargo filter:",len(votacao))

print(votacao["Cargo"].value_counts(dropna=False))


Rows after cargo filter: 9377797
Cargo
Deputado Estadual     5419572
Deputado Federal      3756403
Presidente              81679
Governador              58722
Senador                 50439
Deputado Distrital      10982
Name: count, dtype: int64


### Limpeza e padronização da base eleitoral

Após a seleção dos registros relevantes, a base eleitoral passa por uma etapa de limpeza estrutural. Os nomes das colunas são padronizados para evitar problemas relacionados a espaços ou diferenças de nomenclatura. As variáveis textuais também são tratadas para garantir uniformidade nos valores categóricos. Além disso, atributos que não possuem relevância para a análise são removidos, reduzindo ruído e mantendo apenas as informações necessárias.

### Conversão e tratamento das variáveis numéricas

Nesta etapa, as variáveis originalmente armazenadas como texto, principalmente aquelas relacionadas a códigos, identificadores e quantidade de votos, passam por transformação para formatos numéricos. Esse processo envolve a remoção de caracteres utilizados como separadores de milhar, ajustes de separadores decimais e conversão dos campos para tipos numéricos apropriados. O objetivo é permitir operações matemáticas e análises quantitativas posteriormente.

In [3]:

DROP_COLS = [
    "Data de carga",
    "Ano de eleição"
]
votacao = votacao.drop(columns=DROP_COLS)

votacao_numeric_cols = [
    "Código município",
    "Número candidato",
    "Turno",
    "Zona",
    "Votos válidos",
    "Votos nominais"
]

votacao_numeric_cols = [
    col
    for col in votacao_numeric_cols
    if col in votacao.columns
]

for col in votacao_numeric_cols:
    votacao[col] = clean_numeric_series(votacao[col])

if "Data de carga" in votacao.columns:
    votacao["Data de carga"] = pd.to_datetime(votacao["Data de carga"],errors="coerce")

votacao_object_cols = [
    col
    for col in votacao.select_dtypes(include=["object", "string"]).columns
    if col != "Data de carga"
]

for col in votacao_object_cols:
    mode_values = votacao[col].mode(dropna=True)
    fill_value = mode_values.iloc[0] if not mode_values.empty else "Sem dados"
    votacao[col] = votacao[col].fillna(fill_value)

votacao = votacao.drop_duplicates().reset_index(drop=True)

print("\nRemaining missing values in votacao:")

print(int(votacao.isna().sum().sum()))

votacao["vote_share"] = np.where(votacao["Votos válidos"] > 0,votacao["Votos nominais"] / votacao["Votos válidos"],0)



Remaining missing values in votacao:
0


### Tratamento de valores ausentes

Após a identificação dos valores ausentes, é realizado o processo de imputação para preservar a quantidade de registros disponíveis. Nas variáveis categóricas, os valores faltantes são substituídos pela moda da respectiva coluna, ou seja, pelo valor mais frequente. Essa estratégia permite manter a distribuição original das categorias sem criar novos valores artificiais. Nas variáveis numéricas, é aplicado o tratamento definido no processamento da base para evitar que valores ausentes prejudiquem as análises.

### Criação de novas variáveis derivadas

A partir das variáveis originais, são criadas novas informações derivadas que podem representar melhor o comportamento dos dados. No caso da base eleitoral, é calculada uma variável de participação relativa dos votos, obtida pela relação entre votos recebidos pelo candidato e o total de votos válidos. Essa transformação permite comparar candidatos de diferentes localidades e tamanhos eleitorais de forma proporcional.

### Processamento da base municipal

A base municipal passa por um processo semelhante de preparação. Inicialmente são removidas variáveis consideradas pouco relevantes para o objetivo do estudo, reduzindo a dimensionalidade do conjunto de dados. Em seguida, os atributos textuais e numéricos são padronizados, garantindo consistência nos formatos e facilitando a integração com outras informações utilizadas posteriormente.

### Remoção de inconsistências e duplicidades

Após as transformações iniciais, são realizadas verificações para identificar possíveis problemas nos dados. Registros duplicados são removidos para evitar que algumas observações tenham peso artificialmente maior nas análises. Também são tratados registros sem identificadores válidos, garantindo que cada município ou candidato seja representado de forma única e consistente.

In [4]:
DROP_COLS = [
    "Nome masculino mais popular",
    "Nome feminino mais popular",
    "Sobrenome mais popular",
    "Região intermediária",
    "Região imediata",
    "Mesorregião",
    "Microrregião",
    "Região de Influência",
    "População estimada",
    "População quilombola",
    "População indígena",
    "População exposta ao risco"
]
panorama = panorama.drop(columns=DROP_COLS)

panorama = standardize_missing_values(panorama)
panorama.columns = panorama.columns.str.strip()
panorama = clean_string_columns(panorama)

panorama_numeric_cols = [
    "municipio_id",
    "População no último censo",
    "Densidade demográfica",
    "Salário médio mensal dos trabalhadores formais",
    "Pessoal ocupado em postos de trabalho formais",
    "Percentual da população com rendimento nominal mensal per capita de até 1/2 salário mínimo",
    "Taxa de escolarização de 6 a 14 anos de idade",
    "IDEB – Anos iniciais do ensino fundamental (Rede pública)",
    "IDEB – Anos finais do ensino fundamental (Rede pública)",
    "Matrículas no ensino fundamental",
    "Matrículas no ensino médio",
    "Docentes no ensino fundamental",
    "Docentes no ensino médio",
    "Número de estabelecimentos de ensino fundamental",
    "Número de estabelecimentos de ensino médio",
    "PIB per capita",
    "Índice de Desenvolvimento Humano Municipal (IDHM)",
    "Total de receitas brutas realizadas",
    "Transferências correntes (Percentual em relação às receitas correntes brutas realizadas)",
    "Total de despesas brutas empenhadas",
    "Mortalidade Infantil",
    "Internações por diarreia pelo SUS",
    "Estabelecimentos de Saúde SUS",
    "Área urbanizada",
    "Esgotamento sanitário por rede geral, rede pluvial ou fossa ligada à rede",
    "Arborização de vias públicas",
    "Urbanização de vias públicas",
    "Área da unidade territorial"
]

panorama_numeric_cols = [
    col
    for col in panorama_numeric_cols
    if col in panorama.columns
]

for col in panorama_numeric_cols:
    panorama[col] = clean_numeric_series(panorama[col])

for col in panorama_numeric_cols:
    median_value = panorama[col].median()
    if pd.notna(median_value):
        panorama[col] = panorama[col].fillna(median_value)

panorama_object_cols = [
    col
    for col in panorama.select_dtypes(include=["object", "string"]).columns
    if col != "municipio"
]

for col in panorama_object_cols:
    mode_values = panorama[col].mode(dropna=True)
    fill_value = mode_values.iloc[0] if not mode_values.empty else "Sem dados"
    panorama[col] = panorama[col].fillna(fill_value)

if "municipio_id" in panorama.columns:
    panorama = panorama.dropna(subset=["municipio_id"]).copy()
    panorama = panorama.drop_duplicates(subset=["municipio_id"]).copy()

print(
    "\nRemaining missing values in panorama:"
)

print(
    int(panorama.isna().sum().sum())
)



Remaining missing values in panorama:
0


### Exportação das bases tratadas

Ao final do processo de pré-processamento, as bases resultantes são salvas em novos arquivos contendo apenas os dados limpos e estruturados. Esses arquivos representam a versão final utilizada nas etapas seguintes do projeto, evitando a necessidade de repetir todo o processo de limpeza sempre que uma nova análise for realizada.

In [5]:
votacao.to_csv("votacao_candidato_filtrado_limpo.csv",index=False,encoding="utf-8-sig")

print("\nSaved to votacao_candidato_filtrado_limpo.csv")



Saved to votacao_candidato_filtrado_limpo.csv


In [6]:
panorama.to_csv("dados_municipios_panorama_limpo.csv",index=False,encoding="utf-8-sig")

print("\nSaved to dados_municipios_panorama_limpo.csv")



Saved to dados_municipios_panorama_limpo.csv


### Verificação final da qualidade dos dados

Por fim, são realizadas verificações para confirmar que o processamento foi executado corretamente. São analisadas as dimensões finais das bases, a quantidade de valores ausentes restantes e algumas estatísticas gerais dos dados. Essa etapa garante que os conjuntos finais estejam prontos para serem utilizados nos modelos e análises posteriores.

In [7]:
print("\nFiltered votacao shape:")
print(votacao.shape)

print("\nCleaned panorama shape:")
print(panorama.shape)

print("\nTop missing counts in votacao:")
print(votacao.isna().sum().sort_values(ascending=False).head(10))

print("\nTop missing counts in panorama:")
print(panorama.isna().sum().sort_values(ascending=False).head(10))



Filtered votacao shape:
(9377797, 20)

Cleaned panorama shape:
(5568, 32)

Top missing counts in votacao:
Cargo                0
Código município     0
Cor/raça             0
Estado civil         0
Faixa etária         0
Gênero               0
Grau de instrução    0
Município            0
Nome candidato       0
Número candidato     0
dtype: int64

Top missing counts in panorama:
municipio                                                                                     0
municipio_id                                                                                  0
População no último censo                                                                     0
Densidade demográfica                                                                         0
Salário médio mensal dos trabalhadores formais                                                0
Pessoal ocupado em postos de trabalho formais                                                 0
Percentual da população com rendimento no